# Extraction des Boundary Tokens - CAMeL-BERT

Ce notebook extrait les tokens prédits comme **isnads** (boundary tokens) par CAMeL-BERT du corpus Kitab Uqala.

**Exécution requise:** Google Colab ou environnement avec PyTorch/Transformers


In [ ]:
# Installation des dépendances (si nécessaire)
!pip install transformers torch --quiet

In [ ]:
# Imports
import json
from transformers import AutoTokenizer
from pathlib import Path

# Configuration des chemins
CORPUS_PATH = "data/processed/kitab_uqala_reference_corpus.txt"
PREDICTIONS_PATH = "results/camelbert_kitab_uqala_raw_inference.json"
OUTPUT_PATH = "results/camelbert_boundary_tokens_clean.json"

# Vérifier qu'on est dans le bon répertoire
print(f"Répertoire courant: {Path.cwd()}")
print(f"Fichiers existants:")
print(f"  - Corpus: {Path(CORPUS_PATH).exists()}")
print(f"  - Prédictions: {Path(PREDICTIONS_PATH).exists()}")

In [ ]:
# [1] Charger le tokenizer CAMeL-BERT
print("[1/5] Charger le tokenizer CAMeL-BERT...")
model_name = "CAMeL-Lab/bert-base-arabic-camelbert-msa"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"      ✓ Tokenizer chargé: {model_name}")
print(f"      Type: {type(tokenizer).__name__}")
print(f"      Vocab size: {len(tokenizer)}")

In [ ]:
# [2] Charger le corpus
print("\n[2/5] Charger le corpus...")
with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"      ✓ Corpus chargé")
print(f"        Taille: {len(text):,} caractères")
print(f"        Début: {text[:100]}...")

In [ ]:
# [3] Tokenizer le corpus
print("\n[3/5] Tokenizer le corpus...")
tokens = tokenizer.tokenize(text)

print(f"      ✓ Tokenization complète")
print(f"        Total tokens: {len(tokens):,}")
print(f"        Premiers tokens: {tokens[:10]}")

In [ ]:
# [4] Charger les prédictions du modèle
print("\n[4/5] Charger les prédictions du modèle...")
with open(PREDICTIONS_PATH, 'r', encoding='utf-8') as f:
    predictions_data = json.load(f)

predictions = predictions_data['inference_results']['predictions']
metadata = predictions_data['metadata']

print(f"      ✓ Prédictions chargées")
print(f"        Total prédictions: {len(predictions):,}")
print(f"        Boundary tokens (1s): {predictions_data['inference_results']['boundary_tokens']:,}")
print(f"        Modèle: {metadata['model']}")

In [ ]:
# [5] Vérifier la correspondance
print("\n[5/5] Vérifier la correspondance tokens↔prédictions...")
print(f"        Tokens tokenizer: {len(tokens):,}")
print(f"        Prédictions model: {len(predictions):,}")

if len(tokens) == len(predictions):
    print(f"        ✓ PARFAIT - Correspondance 1:1")
else:
    diff = len(predictions) - len(tokens)
    print(f"        ⚠️  Mismatch détecté: {diff} tokens de différence")
    print(f"        Utilisation du minimum: {min(len(tokens), len(predictions)):,}")
    min_len = min(len(tokens), len(predictions))
    tokens = tokens[:min_len]
    predictions = predictions[:min_len]

In [ ]:
# Extraire les boundary tokens
print("\nExtraction des boundary tokens...")
boundary_tokens = []
boundary_indices = []

for idx, (token, pred) in enumerate(zip(tokens, predictions)):
    if pred == 1:  # isnad token
        boundary_tokens.append(token)
        boundary_indices.append(idx)

print(f"✓ Extraction complète")
print(f"  Boundary tokens trouvés: {len(boundary_tokens):,}")
print(f"  Pourcentage du corpus: {(len(boundary_tokens)/len(tokens))*100:.2f}%")

In [ ]:
# Afficher un aperçu des boundary tokens
print("\n" + "="*80)
print("APERÇU DES BOUNDARY TOKENS (premiers 50)")
print("="*80)

for i, (token, idx) in enumerate(zip(boundary_tokens[:50], boundary_indices[:50]), 1):
    print(f"{i:3d}. Index {idx:6d}: {token}")

print(f"\n... (affiche 50 sur {len(boundary_tokens):,})")

In [ ]:
# Créer les résultats structurés
results = {
    'metadata': {
        'corpus': CORPUS_PATH,
        'corpus_size_chars': len(text),
        'corpus_size_tokens': len(tokens),
        'model': model_name,
        'total_boundary_tokens': len(boundary_tokens),
        'boundary_percentage': round((len(boundary_tokens) / len(tokens)) * 100, 2),
        'extraction_timestamp': '2026-04-21',
    },
    'statistics': {
        'total_tokens': len(tokens),
        'boundary_tokens_count': len(boundary_tokens),
        'non_boundary_tokens': len(tokens) - len(boundary_tokens),
        'boundary_ratio': round(len(boundary_tokens) / len(tokens), 4),
    },
    'boundary_tokens': boundary_tokens,
    'boundary_indices': boundary_indices,
}

print(f"Résultats structurés créés")
print(f"  Clés: {list(results.keys())}")

In [ ]:
# Sauvegarder les résultats
print(f"\nSauvegarde dans {OUTPUT_PATH}...")

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

import os
file_size = os.path.getsize(OUTPUT_PATH) / 1024
print(f"✓ Fichier sauvegardé")
print(f"  Taille: {file_size:.1f} KB")
print(f"  Location: {OUTPUT_PATH}")

In [ ]:
# Résumé final
print("\n" + "="*80)
print("RÉSUMÉ FINAL")
print("="*80)
print(f"\nCorpus:")
print(f"  Caractères: {len(text):,}")
print(f"  Tokens: {len(tokens):,}")
print(f"\nBoundary Tokens (Isnads):")
print(f"  Count: {len(boundary_tokens):,}")
print(f"  Percentage: {results['metadata']['boundary_percentage']}%")
print(f"\nFichier créé:")
print(f"  {OUTPUT_PATH}")
print(f"  Size: {file_size:.1f} KB")
print(f"\n✓ Prêt pour téléchargement et analyse!")

## Télécharger le fichier

Utilisez la commande ci-dessous pour télécharger le fichier généré:

```python
from google.colab import files
files.download('results/camelbert_boundary_tokens_clean.json')
```